# LSTM model for vitals

To configure your system to run this Notebook, from your terminal enter:

```
virtualenv mylstm

source mylstm/bin/activate

python -m pip install numpy scipy matplotlib ipython jupyter pandas sympy nose keras tensorflow scikit-learn

jupyter notebook
```

From Jupyter Notebooks, navigate to your local/clone of this repo and run lstm.ipynb.

In [533]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

## Source data

This data is sourced from Sentinel and stored locally.

In [534]:
df_vitals = pd.read_csv('vitals_combined_interpolated.csv')
df_vitals.rename(columns = {'reading_date':'timestamp'}, inplace = True)
df_vitals['timestamp'] = pd.to_datetime(df_vitals['timestamp'], errors='coerce')

In [535]:
# Drops batches containing less than 30 observations.
df_recordings_per_patient = df_vitals.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]
for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    df_vitals.drop(df_vitals[(df_vitals.patient_uuid == low_patient) & (df_vitals.batch == low_batch)].index, inplace=True)

In [536]:
df_vitals=df_vitals[(df_vitals.timestamp >= '2018-10-01') & (df_vitals.timestamp <= '2020-01-14')]

In [537]:
# Systolic Blood Pressure. 
df_vitals_sbp = df_vitals[['timestamp','patient_uuid','SBP']] 

# Heart rate. 
df_vitals_hr = df_vitals[['timestamp','patient_uuid','HR']]

In [538]:
patients = df_vitals.patient_uuid.unique() # to run with a subset, replace with patients[5:9]
patientsS = patients[5:9]
#patientsS

## LSTM Modeling

This creates an LSTM network for a univariate analysis of vitals passed in as a dataframe with two columns, timestamp and vital sign (eg SBP or HR).

Steps_in defines the look back count (set to 7, here).

Steps_out defines the forward projection count (also set to 7, here).

In [539]:
def split_sequence(sequence, n_steps_in, n_steps_out):
	X, y = list(), list()
	for i in range(len(sequence)):
		# find the end of this pattern
		end_ix = i + n_steps_in
		out_end_ix = end_ix + n_steps_out
		# check if we are beyond the sequence
		if out_end_ix > len(sequence):
			break
		# gather input and output parts of the pattern
		seq_x, seq_y = sequence[i:end_ix], sequence[end_ix:out_end_ix]
		X.append(seq_x)
		y.append(seq_y)
	return np.array(X), np.array(y)

In [540]:
def lstm_model(dataset):
    # split a univariate sequence into samples
    tsData = dataset.copy(deep=True)
    dataset.drop(columns=['timestamp'], inplace=True)
    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset = scaler.fit_transform(dataset)
    
    train_size = len(dataset) - 7
    test_size = 7 
    train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]
    
    train_result = tsData[0:train_size]
    test_result = tsData[train_size:len(tsData)]
    
    # choose a number of time steps
    n_steps_in = 7
    n_steps_out = 7

    # split into samples
    templist = []
    for i in range(len(train)):
        templist.append(train[i][0])

    X, y = split_sequence(templist, n_steps_in, n_steps_out)
    
    templist2 = []
    for j in range(len(test)):
        templist2.append(test[j][0])
    testX, testy = split_sequence(templist2, n_steps_in, n_steps_out)

    # reshape from [samples, timesteps] into [samples, timesteps, features]
    n_features = 1
    X = X.reshape((X.shape[0], X.shape[1], n_features))

    # define model
    model = Sequential()
    model.add(LSTM(100, activation='relu', return_sequences=True, input_shape=(n_steps_in, n_features)))
    model.add(LSTM(100, activation='relu'))
    model.add(Dense(n_steps_out))
    model.compile(optimizer='adam', loss='mse')

    # fit model
    model.fit(X, y, epochs=50, verbose=0)

    # Evaluate vs. test
    testX = test.reshape((1, n_steps_in, n_features))
    testPredict = model.predict(testX) # no memory
    
    # Make prediction of future
    x_input = dataset[len(dataset)-n_steps_in:len(dataset),:] #must be n_steps_in long

    x_input = x_input.reshape((1, n_steps_in, n_features))
    yhat = model.predict(x_input, verbose=0)
    predictions = scaler.inverse_transform(testPredict)

    predData = {'timestamp':tsData[train_size:len(tsData)]['timestamp'].tolist(), 
        tsData.columns[1]:predictions[0].tolist()} # predictions.tolist() [1,2,3,4,5,6,7]
    testPredict_result = pd.DataFrame(predData)
    
    ts = tsData['timestamp'][len(tsData)-1:len(tsData)]
    dr = pd.date_range(ts.values[0]+ pd.Timedelta('1 day'), periods=7, freq='D')
    futures = scaler.inverse_transform(yhat)    
    futData = {'timestamp':dr, 
        tsData.columns[1]:futures[0].tolist()} # futures.tolist()
    yhat_result = pd.DataFrame(futData)
    
    return train_result, test_result, testPredict_result, yhat_result

In [541]:
df_select_uuid = df_vitals_sbp.loc[df_vitals_sbp['patient_uuid'] == 1523219] #1127912 73939445 1164638
df_select_uuid_hr = df_vitals_hr.loc[df_vitals_hr['patient_uuid'] == 1523219]
df_SBP = df_select_uuid[['timestamp','SBP']]
df_HR = df_select_uuid_hr[['timestamp','HR']]

In [542]:
# Get the results of the fit model
tr, te, tP, yh = lstm_model(df_SBP)
trHR, teHR, tPHR, yhHR = lstm_model(df_HR)

In [543]:
pd.set_option('mode.chained_assignment', None)

# Runs LSTM model on all patients and collects data on accuracy.
HR_mape = []
HR_mae = []
HR_rsme = []
SBP_mape = []
SBP_mae = []
SBP_rsme = []
i = 0

patientsS = patients[0:8]
print('patient: ',patientsS)

for patient in patientsS: #patientsS for a subset    
    print('patient ', i, ': ', patient)
    
    df_select_uuid = df_vitals.loc[df_vitals['patient_uuid'] == patient]
    df_SBP = df_select_uuid[['timestamp','SBP']]
    df_HR = df_select_uuid[['timestamp','HR']]    
    
    results_trainS, results_testS, results_forecastS, results_futureS = lstm_model(df_SBP)
    results_trainH, results_testH, results_forecastH, results_futureH = lstm_model(df_HR)
    
    results_train = pd.DataFrame()
    results_train['timestamp'] = results_trainS['timestamp'].values
    results_train['SBP'] = results_trainS['SBP'].values
    results_train['HR'] = results_trainH['HR'].values
    
    results_test = pd.DataFrame()
    results_test['timestamp'] = results_testS['timestamp'].values
    results_test['SBP'] = results_testS['SBP'].values
    results_test['HR'] = results_testH['HR'].values

    results_forecast = pd.DataFrame()
    results_forecast['timestamp'] = results_forecastS['timestamp'].values
    results_forecast['SBP'] = results_forecastS['SBP'].values
    results_forecast['HR'] = results_forecastH['HR'].values
    
    results_future = pd.DataFrame()
    results_future['timestamp'] = results_futureS['SBP'].values
    results_future['SBP'] = results_futureS['SBP'].values
    results_future['HR'] = results_futureH['HR'].values
    
    i = i + 1    
    
    #print(patient, ' results_forecast: ', results_forecast.head())
    #print(patient, ' results_test: ', results_test.head())
    
    accuracy_indicators = forecast_accuracy(results_forecast, results_test)
    HR_mape.append(accuracy_indicators[0][0])
    HR_mae.append(accuracy_indicators[1][0])
    HR_rsme.append(accuracy_indicators[2][0])
    SBP_mape.append(accuracy_indicators[0][1])
    SBP_mae.append(accuracy_indicators[1][1])
    SBP_rsme.append(accuracy_indicators[2][1])


patient:  [1127912 1164638 1166047 1193821 1231804 1519674 1523219 2075791]
patient  0 :  1127912
patient  1 :  1164638
patient  2 :  1166047
patient  3 :  1193821
patient  4 :  1231804
patient  5 :  1519674
patient  6 :  1523219
patient  7 :  2075791


In [544]:
# Creates and populates dataframe containing data on accuracy of the models for each patient.
df_model = pd.DataFrame(columns = ['patient_uuid', 'HR_mape', 'HR_mae', 'HR_rsme', 'SBP_mape', 'SBP_mae', 'SBP_rsme'])
df_model.patient_uuid = patientsS #patients
df_model.HR_mape = HR_mape
df_model.HR_mae = HR_mae
df_model.HR_rsme = HR_rsme
df_model.SBP_mape = SBP_mape
df_model.SBP_mae = SBP_mae
df_model.SBP_rsme = SBP_rsme

#df_model.head()

In [545]:
def forecast_accuracy(forecast, actual):
    """
     Calculates accuracy by calculating mean absolute percentage error, mean absolute error, 
     and root-mean-squared error.
    
    forecast: forecast dataframe containing the results from a model.
    actual  : test set of a particular patient, with same dates as the forecast dataframe.
    
    output  : mape: mean absolute percentage error
              mae : mean absolute error
              rsme: root mean square error
    """
    #forecast = forecast.values
    
    forecast = forecast.set_index('timestamp')
    actual = actual.set_index('timestamp')
    
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, mae, rmse

In [546]:
# save model results to file
df_model.to_csv('df_model.csv')
df = pd.read_csv('df_model.csv')

In [547]:
df_lstm_metrics = pd.DataFrame((df_model[['HR_mape', 'HR_mae', 'HR_rsme', 'SBP_mape','SBP_mae','SBP_rsme']].mean()), 
                               columns=['LSTM'])
#df_lstm_metrics

## Flag setting

Push results through the flag setting model as documented by Agustin in VAR.

In [548]:
def compare_flags(df_train, df_test, df_forecast):
    """
    Compares the results of the flag-raising algorithm.
    
    df_train   : training set of the timeseries for a particular patient.
    df_test    : test set of the timeseries for a particular patient.
    df_forecast: forecast of the timeseries for a particular patient.
    
    output     : list containing HR_error, SBP_error, algorithm accuracy, 
                 predicted flags and actual flags.
    """
    flag_comparison = []
    current = df_train[['HR','SBP']][-23:].copy()
    future = df_test[['HR','SBP']].copy()
    forecast = df_forecast.copy()
    forecast.columns = ['HR', 'SBP']
    actual = pd.concat([current, future])
    pred = pd.concat([current, forecast])
    
    actual_flags = flags(actual)
    pred_flags = flags(pred)
    
    HR_error = np.abs(pred_flags[0][0] - actual_flags[0][0]) / actual_flags[0][0]
    SBP_error = np.abs(pred_flags[0][1] - actual_flags[0][1]) / actual_flags[0][1]

    for i in range(len(actual_flags)):
        flag_comparison.append(actual_flags[i] == pred_flags[i])
    
    algo_accuracy = np.sum(flag_comparison[1:]) / 8
    
    result = [HR_error, SBP_error, algo_accuracy] + actual_flags[1:] + pred_flags[1:]
    
    return result

In [549]:
def flags(df, report=False):
    """
    Flag-raising algorithm based on health deterioration rules.
    
    df    : 30-day timeseries pandas dataframe.
    report: True if written report requested. False if status of each flag is wanted.
    
    output: returns a list containing the average value of HR and SBP during the last ten days of the time series,
            and the different flags that are raised based on the conditions above.
    """
    rising_HR = False
    lowering_HR = False
    rising_SBP = False
    high_BP = False
    
    first_ten = np.mean(df[:10])
    last_ten = np.mean(df[-10:])

    deltas = (last_ten - first_ten)
    
    if deltas[0] >= 10:
        rising_HR = True
        flag_message_1 = 'Detected rising heart rate. \n'
    elif deltas[0] <= -10:
        lowering_HR = True
        flag_message_1 = 'Detected decreasing heart rate. \n'
    else:
        flag_message_1 = 'No changes in heart rate detected. \n'
        
    if deltas[1] >= 5:
        rising_SBP = True
        flag_message_2 = 'Detected rising blood pressure (SBP). \n'
    else:
        flag_message_2 = 'No changes in blood pressure detected. \n'

    min_HR = np.sum(df.HR < 55)
    max_HR = np.sum(df.HR > 100)

    min_SBP = np.sum(df.SBP < 90)
    max_SBP = np.sum(df.SBP > 180)
    
    flag_message_3 = 'Heart rate lower than 55 bpm detected ' + str(min_HR) + ' times. \n'
    flag_message_4 = 'Heart rate greater than 100 bpm detected ' + str(max_HR) + ' times. \n'
    flag_message_5 = 'Systolic blood pressure (SBP) lower than 90 bpm detected ' + str(min_SBP) + ' times. \n'
    flag_message_6 = 'Systolic blood pressure (SBP) greater than 180 bpm detected ' + str(max_SBP) + ' times. \n'
    flag_message_7 = ''
    
    high_SBP = (df.SBP > 140)
    
    if np.sum(high_SBP[-15:]) > 7:
        high_BP = True
        flag_message_7 = 'High blood pressure detected.'
    if report == True:
        if high_BP == True:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6 + flag_message_7 
        else:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6
        print(message)
    else:
        return [last_ten, rising_HR, lowering_HR, rising_SBP, min_HR, max_HR, min_SBP, max_SBP, high_BP]

In [550]:
future = results_future.set_index('timestamp')
train = results_train.set_index('timestamp')
test  = results_test.set_index('timestamp')
forecast = results_forecast.set_index('timestamp')

In [551]:
flag_columns = ['patient_uuid', 'HR_error', 'SBP_error', 'algo_accuracy', 'actual_rising_HR', 'actual_lowering_HR',
                'actual_rising_SBP', 'actual_min_HR', 'actual_max_HR', 'actual_min_SBP', 'actual_max_SBP', 'actual_high_BP',
                'pred_rising_HR', 'pred_lowering_HR', 'pred_rising_SBP', 'pred_min_HR', 'pred_max_HR', 'pred_min_SBP',
                'pred_max_SBP', 'pred_high_BP']
data = []
for patient in patientsS: #patients
    df_train, df_test, df_forecast, df_future = train, test, forecast, future #var_model(patient)
    flag_row = compare_flags(df_train, df_test, df_forecast)
    flag_row = [patient] + flag_row
    data.append(flag_row)
df_flags = pd.DataFrame(data, columns=flag_columns)
df_flags = df_flags[['patient_uuid', 'HR_error', 'SBP_error', 'algo_accuracy', 'pred_rising_HR', 'actual_rising_HR',
                     'pred_lowering_HR', 'actual_lowering_HR', 'pred_rising_SBP', 'actual_rising_SBP', 'pred_high_BP',
                     'actual_high_BP', 'pred_min_HR', 'actual_min_HR', 'pred_max_HR', 'actual_max_HR', 'pred_min_SBP', 
                     'actual_min_SBP', 'pred_max_SBP', 'actual_max_SBP']]

In [552]:
df_flags

,patient_uuid,HR_error,SBP_error,algo_accuracy,pred_rising_HR,actual_rising_HR,pred_lowering_HR,actual_lowering_HR,pred_rising_SBP,actual_rising_SBP,pred_high_BP,actual_high_BP,pred_min_HR,actual_min_HR,pred_max_HR,actual_max_HR,pred_min_SBP,actual_min_SBP,pred_max_SBP,actual_max_SBP
0,1127912,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
1,1164638,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
2,1166047,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
3,1193821,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
4,1231804,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
5,1519674,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
6,1523219,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0
7,2075791,0.996408,0.404227,0.5,True,False,False,False,False,False,False,False,10,16,7,0,7,0,0,0


In [553]:
df_flags.to_csv('lstm_flag_stats.csv', index=False)